In [ ]:
import os
import numpy as np
import pandas as pd


def extract_pfc_subpathways_with_length(
    file_path,
    outpath,
    target_regions=("PL", "ORB", "ACA", "ILA", "FRP"),
    exclude_groups=("STR", "OLF", "CTXsp", "amc"),
    coordinate_cols=("x", "y", "z"),
    coordinate_scale=1.0,
    decimals=2
):
    """
    提取从胞体到PFC末梢的投射路径，并标注每一次连续经过
    某个脑区时的轴突长度。

    例如：
    BLA (125.30 μm) -> EP (8.60 μm) -> PIR (42.10 μm)
    -> EP (16.70 μm) -> ORB (78.20 μm)

    注意：
    如果同一脑区被轴突多次进入，每一次进入将单独统计，
    不会将不同位置的片段合并。

    Parameters
    ----------
    file_path : str
        输入CSV文件。

    outpath : str
        输出文件夹。

    target_regions : tuple/list
        PFC目标脑区。

    exclude_groups : tuple/list
        不显示在最终路径中的脑区。
        设为 () 或 [] 表示不排除任何脑区。

    coordinate_cols : tuple
        坐标列名称，默认 ("x", "y", "z")。

    coordinate_scale : float
        坐标到μm的转换系数。

        如果x、y、z本身就是μm，设为1.0。
        如果坐标单位是像素，应设置为实际的μm/像素。

    decimals : int
        路径长度保留的小数位数。
    """

    filename = os.path.splitext(
        os.path.basename(file_path)
    )[0]

    dfn = pd.read_csv(file_path, index_col=0)

    # ============================================================
    # 1. 检查必要列
    # ============================================================
    required_cols = {
        "ID",
        "parent",
        "name_use",
        "area_name",
        "terminal",
        *coordinate_cols
    }

    missing_cols = required_cols - set(dfn.columns)

    if missing_cols:
        raise ValueError(
            f"输入文件缺少以下必要列：{sorted(missing_cols)}"
        )

    # 统一数据类型
    dfn["ID"] = pd.to_numeric(
        dfn["ID"],
        errors="coerce"
    )

    dfn["parent"] = pd.to_numeric(
        dfn["parent"],
        errors="coerce"
    )

    dfn = dfn.dropna(subset=["ID"]).copy()
    dfn["ID"] = dfn["ID"].astype(int)
    dfn["parent"] = dfn["parent"].astype("Int64")

    for col in coordinate_cols:
        dfn[col] = pd.to_numeric(
            dfn[col],
            errors="coerce"
        )

    # ============================================================
    # 2. 建立快速查找字典
    # ============================================================
    parent_map = dict(
        zip(dfn["ID"], dfn["parent"])
    )

    group_map = dict(
        zip(dfn["ID"], dfn["name_use"])
    )

    area_detail_map = dict(
        zip(dfn["ID"], dfn["area_name"])
    )

    coordinate_map = {
        int(row["ID"]): np.array(
            [row[col] for col in coordinate_cols],
            dtype=float
        )
        for _, row in dfn.iterrows()
    }

    # ============================================================
    # 3. 寻找位于PFC目标脑区的末梢
    # ============================================================
    dft = dfn.loc[
        (dfn["terminal"] == 1) &
        (dfn["name_use"].isin(target_regions))
    ].copy()

    if dft.empty:
        print(
            f"未在 {filename} 中找到投射至 "
            f"{list(target_regions)} 的末梢节点。"
        )
        return None

    terminal_ids = dft["ID"].tolist()

    all_path_ids = set()
    path_trajectories = []
    segment_records = []

    # ============================================================
    # 4. 依次处理每个PFC末梢
    # ============================================================
    for terminal_id in terminal_ids:

        # --------------------------------------------------------
        # 4.1 从终末回溯到胞体
        # --------------------------------------------------------
        current_id = terminal_id
        reverse_path_ids = []
        visited_ids = set()

        while True:

            # 防止parent关系出现循环
            if current_id in visited_ids:
                print(
                    f"警告：末梢 {terminal_id} 的路径中检测到循环，"
                    f"循环节点为 {current_id}。"
                )
                break

            visited_ids.add(current_id)
            reverse_path_ids.append(current_id)
            all_path_ids.add(current_id)

            parent_id = parent_map.get(current_id)

            # 到达SWC根节点
            if pd.isna(parent_id):
                break

            parent_id = int(parent_id)

            if parent_id == -1:
                break

            if parent_id not in parent_map:
                print(
                    f"警告：节点 {current_id} 的父节点 "
                    f"{parent_id} 不存在。"
                )
                break

            current_id = parent_id

        # 转换为：胞体 -> 终末
        path_ids = reverse_path_ids[::-1]

        if len(path_ids) == 0:
            continue

        # --------------------------------------------------------
        # 4.2 将连续经过同一脑区的节点划分为独立片段
        # --------------------------------------------------------
        region_segments = []

        first_node_id = path_ids[0]
        first_region = str(
            group_map.get(first_node_id, "Unknown")
        )

        current_segment = {
            "region": first_region,
            "length_um": 0.0,
            "start_node_id": first_node_id,
            "end_node_id": first_node_id,
            "node_count": 1
        }

        for parent_id, child_id in zip(
            path_ids[:-1],
            path_ids[1:]
        ):
            parent_coordinate = coordinate_map.get(parent_id)
            child_coordinate = coordinate_map.get(child_id)

            # 计算两个相邻SWC节点之间的三维欧氏距离
            if (
                parent_coordinate is None or
                child_coordinate is None or
                np.any(np.isnan(parent_coordinate)) or
                np.any(np.isnan(child_coordinate))
            ):
                segment_length_um = 0.0

            else:
                segment_length_um = float(
                    np.linalg.norm(
                        child_coordinate - parent_coordinate
                    )
                ) * coordinate_scale

            child_region = str(
                group_map.get(child_id, "Unknown")
            )

            if child_region == current_segment["region"]:
                # 仍然位于当前脑区
                current_segment["length_um"] += segment_length_um
                current_segment["end_node_id"] = child_id
                current_segment["node_count"] += 1

            else:
                # 离开当前脑区：保存当前片段
                region_segments.append(current_segment)

                # 从新的脑区开始一个新片段
                current_segment = {
                    "region": child_region,
                    "length_um": segment_length_um,
                    "start_node_id": child_id,
                    "end_node_id": child_id,
                    "node_count": 1
                }

        # 保存最后一个脑区片段
        region_segments.append(current_segment)

        # --------------------------------------------------------
        # 4.3 给每一次进入同一脑区进行编号
        #
        # 例如：
        # EP第一次进入：visit_number = 1
        # EP第二次进入：visit_number = 2
        # --------------------------------------------------------
        region_visit_counter = {}

        for path_order, region_segment in enumerate(
            region_segments,
            start=1
        ):
            region = region_segment["region"]

            region_visit_counter[region] = (
                region_visit_counter.get(region, 0) + 1
            )

            region_segment["visit_number"] = (
                region_visit_counter[region]
            )

            region_segment["path_order"] = path_order

        # --------------------------------------------------------
        # 4.4 排除指定脑区
        #
        # 注意：这里只是不在最终路径中显示，
        # 不会把前后相同脑区的独立片段重新合并。
        # --------------------------------------------------------
        displayed_segments = [
            segment
            for segment in region_segments
            if segment["region"] not in exclude_groups
        ]

        # 不带长度的路径
        trajectory_without_length = " -> ".join(
            segment["region"]
            for segment in displayed_segments
        )

        # 带每段长度的路径
        trajectory_with_length = " -> ".join(
            (
                f"{segment['region']} "
                f"({segment['length_um']:.{decimals}f} μm)"
            )
            for segment in displayed_segments
        )

        total_path_length = sum(
            segment["length_um"]
            for segment in region_segments
        )

        displayed_path_length = sum(
            segment["length_um"]
            for segment in displayed_segments
        )

        terminal_area_detail = area_detail_map.get(
            terminal_id,
            "Unknown"
        )

        terminal_group = group_map.get(
            terminal_id,
            "Unknown"
        )

        path_trajectories.append({
            "Terminal_ID": terminal_id,
            "Terminal_Area_Detail": terminal_area_detail,
            "Terminal_Group": terminal_group,
            "Trajectory": trajectory_without_length,
            "Trajectory_with_Length": trajectory_with_length,
            "Number_of_Region_Segments": len(region_segments),
            "Number_of_Displayed_Segments": len(
                displayed_segments
            ),
            "Total_Path_Length_um": total_path_length,
            "Displayed_Path_Length_um": displayed_path_length
        })

        # --------------------------------------------------------
        # 4.5 保存每个连续脑区片段的详细信息
        # --------------------------------------------------------
        for segment in region_segments:

            is_excluded = (
                segment["region"] in exclude_groups
            )

            segment_records.append({
                "Terminal_ID": terminal_id,
                "Terminal_Area_Detail": terminal_area_detail,
                "Terminal_Group": terminal_group,
                "Path_Order": segment["path_order"],
                "Region": segment["region"],
                "Region_Visit_Number": segment["visit_number"],
                "Axon_Length_um": segment["length_um"],
                "Start_Node_ID": segment["start_node_id"],
                "End_Node_ID": segment["end_node_id"],
                "Node_Count": segment["node_count"],
                "Is_Excluded": is_excluded
            })

    # ============================================================
    # 5. 提取所有PFC末梢对应的SWC子树
    # ============================================================
    dff = dfn.loc[
        dfn["ID"].isin(all_path_ids)
    ].copy()

    dff["path_node_group"] = dff["name_use"]

    dff["is_pfc_target_terminal"] = (
        dff["ID"].isin(terminal_ids)
    )

    # ============================================================
    # 6. 转为DataFrame
    # ============================================================
    df_trajectory = pd.DataFrame(path_trajectories)
    df_segments = pd.DataFrame(segment_records)

    # 长度保留指定小数位
    if not df_trajectory.empty:
        df_trajectory["Total_Path_Length_um"] = (
            df_trajectory["Total_Path_Length_um"].round(decimals)
        )

        df_trajectory["Displayed_Path_Length_um"] = (
            df_trajectory["Displayed_Path_Length_um"].round(decimals)
        )

    if not df_segments.empty:
        df_segments["Axon_Length_um"] = (
            df_segments["Axon_Length_um"].round(decimals)
        )

    # ============================================================
    # 7. 保存结果
    # ============================================================
    os.makedirs(outpath, exist_ok=True)

    subtree_outfile = os.path.join(
        outpath,
        f"{filename}_PFC_subtree.csv"
    )

    trajectory_outfile = os.path.join(
        outpath,
        f"{filename}_PFC_trajectories_with_length.csv"
    )

    segment_outfile = os.path.join(
        outpath,
        f"{filename}_PFC_region_segments.csv"
    )

    dff.to_csv(
        subtree_outfile,
        index=False,
        encoding="utf-8-sig"
    )

    df_trajectory.to_csv(
        trajectory_outfile,
        index=False,
        encoding="utf-8-sig"
    )

    df_segments.to_csv(
        segment_outfile,
        index=False,
        encoding="utf-8-sig"
    )

    # ============================================================
    # 8. 打印结果
    # ============================================================
    print(f"处理完成：{filename}")
    print(f"- PFC子树：{subtree_outfile}")
    print(f"- 带长度的完整路径：{trajectory_outfile}")
    print(f"- 每个脑区片段的详细长度：{segment_outfile}")

    print("\n=== 带脑区片段长度的路径示例 ===")

    for _, row in df_trajectory.head(5).iterrows():
        print(
            f"\n末梢 [{row['Terminal_Area_Detail']}] "
            f"(ID: {row['Terminal_ID']}):"
        )
        print(f"  {row['Trajectory_with_Length']}")

    return dff, df_trajectory, df_segments

In [ ]:
import glob
from tqdm import tqdm

In [ ]:
files = glob.glob(r"J:\BLA_four_types\csv_terminal\*.csv")
# os.makedirs(r"J:\BLA_four_types\csv_PFC_new")
outpath = r"J:\BLA_three_types\csv_PFC_ways_0920"
for file in tqdm(files):
    try:
        dff, df_trajectory, df_segments = (
                extract_pfc_subpathways_with_length(
                    file_path=file,
                    outpath=outpath,
                    coordinate_cols=("AP", "DV", "ML"),
                    coordinate_scale=1.0,
                    decimals=2
                )
            )
    except:
        print(file)
        


In [ ]:
files = glob.glob(r"J:\BLA_three_types\csv_PFC_ways_0920\raw\*with_length*")

['J:\\BLA_three_types\\csv_PFC_ways_0920\\raw\\251034_003_Vglut1_PFC_trajectories_with_length.csv',
 'J:\\BLA_three_types\\csv_PFC_ways_0920\\raw\\251034_004_Vglut1_PFC_trajectories_with_length.csv',
 'J:\\BLA_three_types\\csv_PFC_ways_0920\\raw\\251034_010_Vglut1_PFC_trajectories_with_length.csv',
 'J:\\BLA_three_types\\csv_PFC_ways_0920\\raw\\251039_020_Vglut1_PFC_trajectories_with_length.csv',
 'J:\\BLA_three_types\\csv_PFC_ways_0920\\raw\\251039_021_Vglut1_PFC_trajectories_with_length.csv',
 'J:\\BLA_three_types\\csv_PFC_ways_0920\\raw\\251039_029_Vglut1_PFC_trajectories_with_length.csv',
 'J:\\BLA_three_types\\csv_PFC_ways_0920\\raw\\251039_033_Vglut1_PFC_trajectories_with_length.csv',
 'J:\\BLA_three_types\\csv_PFC_ways_0920\\raw\\251034_001_Vglut1_PFC_trajectories_with_length.csv']

In [14]:
import os
import re
import glob
import pandas as pd
from tqdm import tqdm
from openpyxl.styles import Alignment


# ===================== Parameters =====================
INPUT_PATTERN = r"J:\BLA_three_types\csv_PFC_ways_0920\raw\*with_length*.csv"
OUTPUT_EXCEL = r"J:\BLA_three_types\csv_PFC_ways_0920\PFC_simple_pathways_merged.xlsx"
PFC_REGIONS = {"PL", "ORB", "ACA", "ILA", "FRP"}


def split_path(path):
    """Split a trajectory written with either '->' or '→'."""
    if pd.isna(path):
        return []
    return [x.strip() for x in re.split(r"\s*(?:->|→)\s*", str(path)) if x.strip()]


def region_name(text):
    """Convert 'ORB (12.30 μm)' to 'ORB'."""
    return re.sub(
        r"\s*\([^)]*(?:μm|um)[^)]*\)\s*$", "", str(text), flags=re.IGNORECASE
    ).strip()


def simplify_path(plain_path, length_path):
    """Keep the trajectory through the first PFC region."""
    plain = split_path(plain_path)
    lengths = split_path(length_path)

    first_pfc_index = next(
        (i for i, item in enumerate(plain) if region_name(item) in PFC_REGIONS),
        None,
    )
    if first_pfc_index is None:
        return None, None, None

    first_pfc = region_name(plain[first_pfc_index])
    simple_structure = " -> ".join(plain[: first_pfc_index + 1])

    # Normally the two trajectory columns have exactly the same number of regions.
    if len(plain) == len(lengths):
        simple = lengths[: first_pfc_index + 1]
    else:
        length_index = next(
            (i for i, item in enumerate(lengths) if region_name(item) in PFC_REGIONS),
            None,
        )
        if length_index is None:
            return None, None, None
        simple = lengths[: length_index + 1]

    return simple_structure, " -> ".join(simple), first_pfc


def neuron_name(file_path):
    """Remove the generated suffix from the CSV filename."""
    name = os.path.splitext(os.path.basename(file_path))[0]
    for suffix in (
        "_PFC_trajectories_with_length",
        "_PFC_clean_trajectories_with_length",
        "_trajectories_with_length",
        "_with_length",
    ):
        if name.endswith(suffix):
            return name[: -len(suffix)]
    return name


files = sorted(glob.glob(INPUT_PATTERN))
if not files:
    raise FileNotFoundError(f"No input CSV files found: {INPUT_PATTERN}")

records = []
failed = []

for file_path in tqdm(files, desc="Processing"):
    try:
        df = pd.read_csv(file_path)
        required = {"Trajectory", "Trajectory_with_Length"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"Missing columns: {sorted(missing)}")

        full_path = os.path.abspath(file_path)
        name = neuron_name(file_path)

        for _, row in df.iterrows():
            simple_structure, simple_path, first_pfc = simplify_path(
                row["Trajectory"], row["Trajectory_with_Length"]
            )
            if simple_path is None:
                continue

            records.append(
                {
                    # Keep the complete original CSV path as requested.
                    "Pathway_ID": full_path,
                    "Filename": name,
                    "First_PFC_Region": first_pfc,
                    "Terminal_ID": row.get("Terminal_ID", ""),
                    "Terminal_Area_Detail": row.get("Terminal_Area_Detail", ""),
                    "Terminal_Group": row.get("Terminal_Group", ""),
                    # Used to identify pathways without letting length differences
                    # split an otherwise identical anatomical pathway.
                    "Pathway_Structure": simple_structure,
                    # Each region is followed by the axon length of that segment.
                    "Simple_Trajectory": simple_path,
                }
            )
    except Exception as exc:
        failed.append((file_path, str(exc)))

if not records:
    raise ValueError("No valid PFC trajectories were extracted.")

result = pd.DataFrame(records)

# After truncation at the first PFC region, only identical complete region
# sequences are considered the same pathway. Keep one representative row.
pathway_counts = result.groupby("Pathway_Structure").size().rename("Merged_Record_Count")
result = result.drop_duplicates("Pathway_Structure", keep="first").copy()
result = result.merge(pathway_counts, on="Pathway_Structure", how="left")
result.insert(0, "Pathway_Group", [f"Pathway_{i:02d}" for i in range(1, len(result) + 1)])

result = result[
    [
        "Pathway_Group",
        "Pathway_ID",
        "Filename",
        "First_PFC_Region",
        "Terminal_ID",
        "Terminal_Area_Detail",
        "Terminal_Group",
        "Pathway_Structure",
        "Simple_Trajectory",
        "Merged_Record_Count",
    ]
].reset_index(drop=True)

os.makedirs(os.path.dirname(OUTPUT_EXCEL), exist_ok=True)

with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    result.to_excel(writer, sheet_name="Pathway_Summary", index=False)
    ws = writer.sheets["Pathway_Summary"]
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    widths = {
        "A": 18, "B": 100, "C": 28, "D": 18, "E": 14,
        "F": 24, "G": 18, "H": 90, "I": 120, "J": 22
    }
    for column, width in widths.items():
        ws.column_dimensions[column].width = width
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical="top", wrap_text=True)

print(f"Done: {OUTPUT_EXCEL}")
print(f"Input files: {len(files)}")
print(f"Output rows: {len(result)}")
print(f"Pathways: {result['Pathway_Group'].nunique()}")

if failed:
    print("\nFailed files:")
    for file_path, error in failed:
        print(f"- {file_path}: {error}")


Processing: 100%|██████████| 8/8 [00:00<00:00, 155.58it/s]

Done: J:\BLA_three_types\csv_PFC_ways_0920\PFC_simple_pathways_merged.xlsx
Input files: 8
Output rows: 19
Pathways: 19


In [10]:
import os
import re
import glob
import pandas as pd
from tqdm import tqdm
from openpyxl.styles import Alignment


# ============================================================
# 参数
# ============================================================

files = glob.glob(
    r"J:\BLA_three_types\csv_PFC_ways_0920\raw\*with_length*.csv"
)

output_excel = (
    r"J:\BLA_three_types\csv_PFC_ways_0920"
    r"\PFC_Pathway_Summary.xlsx"
)

pfc_regions = {"PL", "ORB", "ACA", "ILA", "FRP"}


# ============================================================
# 辅助函数
# ============================================================

def split_path(path):
    """拆分路径，同时支持 -> 和 →。"""

    if pd.isna(path):
        return []

    return [
        x.strip()
        for x in re.split(r"\s*(?:->|→)\s*", str(path))
        if x.strip()
    ]


def get_region(text):
    """
    去除axon length，只返回脑区名称。

    ORB (12.30 μm) -> ORB
    """

    return re.sub(
        r"\s*\([^)]*(?:μm|um)[^)]*\)\s*$",
        "",
        str(text),
        flags=re.IGNORECASE
    ).strip()


def simplify_path(
    trajectory,
    trajectory_with_length
):
    """
    只保留到第一次出现PFC脑区为止。

    返回：
    1. 不带长度的路径，用于Pathway分组
    2. 带长度的路径，用于最终显示
    3. 第一个PFC脑区
    """

    plain = split_path(trajectory)
    with_length = split_path(
        trajectory_with_length
    )

    # 找到第一个PFC脑区
    first_pfc_index = None

    for i, region in enumerate(plain):
        if get_region(region) in pfc_regions:
            first_pfc_index = i
            break

    if first_pfc_index is None:
        return None, None, None

    simple_plain = plain[
        :first_pfc_index + 1
    ]

    first_pfc = get_region(
        plain[first_pfc_index]
    )

    # 普通路径和带长度路径数量一致
    if len(plain) == len(with_length):

        simple_with_length = with_length[
            :first_pfc_index + 1
        ]

    else:
        # 数量不一致时，在带长度路径中重新定位
        length_index = None

        for i, region in enumerate(with_length):
            if get_region(region) in pfc_regions:
                length_index = i
                break

        if length_index is None:
            return None, None, None

        simple_with_length = with_length[
            :length_index + 1
        ]

    return (
        " -> ".join(simple_plain),
        " -> ".join(simple_with_length),
        first_pfc
    )


def get_filename(file):
    """提取神经元名称。"""

    name = os.path.splitext(
        os.path.basename(file)
    )[0]

    for suffix in [
        "_PFC_trajectories_with_length",
        "_PFC_clean_trajectories_with_length",
        "_trajectories_with_length",
        "_with_length"
    ]:
        if name.endswith(suffix):
            name = name[:-len(suffix)]
            break

    return name


# ============================================================
# 读取并处理所有文件
# ============================================================

if not files:
    raise FileNotFoundError(
        "没有找到包含 with_length 的CSV文件。"
    )

results = []
failed_files = []

for file in tqdm(files, desc="Processing"):

    try:
        df = pd.read_csv(file)

        required = {
            "Trajectory",
            "Trajectory_with_Length"
        }

        missing = required - set(df.columns)

        if missing:
            raise ValueError(
                f"缺少列：{sorted(missing)}"
            )

        filename = get_filename(file)
        full_path = os.path.abspath(file)

        for _, row in df.iterrows():

            simple_plain, simple_length, first_pfc = (
                simplify_path(
                    row["Trajectory"],
                    row["Trajectory_with_Length"]
                )
            )

            if simple_plain is None:
                continue

            results.append({
                # 相同无长度路径用于分组
                "_Pathway_Key": simple_plain,

                # 完整源文件路径
                "Pathway_ID": full_path,

                "Filename": filename,

                "Terminal_ID": row.get(
                    "Terminal_ID", ""
                ),

                "Terminal_Area_Detail": row.get(
                    "Terminal_Area_Detail", ""
                ),

                "Terminal_Group": row.get(
                    "Terminal_Group", ""
                ),

                "First_PFC_Region": first_pfc,

                # 最终显示带axon length的路径
                "Simple_Trajectory": simple_length
            })

    except Exception as error:

        failed_files.append(
            (file, str(error))
        )

        print(
            f"\n处理失败：{file}\n"
            f"原因：{error}"
        )


if not results:
    raise ValueError(
        "没有提取到有效的PFC路径。"
    )


# ============================================================
# 生成Pathway分组
# ============================================================

result_df = pd.DataFrame(results)

unique_pathways = (
    result_df["_Pathway_Key"]
    .drop_duplicates()
    .tolist()
)

pathway_map = {
    path: f"Pathway_{i:02d}"
    for i, path in enumerate(
        unique_pathways,
        start=1
    )
}

result_df.insert(
    0,
    "Pathway_Group",
    result_df["_Pathway_Key"].map(pathway_map)
)

# 删除内部使用的无长度路径
result_df.drop(
    columns="_Pathway_Key",
    inplace=True
)

# 排序
result_df["_sort"] = (
    result_df["Pathway_Group"]
    .str.extract(r"(\d+)", expand=False)
    .astype(int)
)

result_df = (
    result_df
    .sort_values(
        ["_sort", "Filename", "Terminal_ID"]
    )
    .drop(columns="_sort")
    .reset_index(drop=True)
)


# ============================================================
# 保存Excel：只生成Pathway_Summary
# ============================================================

os.makedirs(
    os.path.dirname(output_excel),
    exist_ok=True
)

with pd.ExcelWriter(
    output_excel,
    engine="openpyxl"
) as writer:

    result_df.to_excel(
        writer,
        sheet_name="Pathway_Summary",
        index=False
    )

    ws = writer.sheets["Pathway_Summary"]

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    widths = {
        "A": 18,    # Pathway_Group
        "B": 100,   # Pathway_ID
        "C": 30,    # Filename
        "D": 15,    # Terminal_ID
        "E": 25,    # Terminal_Area_Detail
        "F": 20,    # Terminal_Group
        "G": 20,    # First_PFC_Region
        "H": 120    # Simple_Trajectory
    }

    for column, width in widths.items():
        ws.column_dimensions[column].width = width

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )


# ============================================================
# 完成
# ============================================================

print("\n处理完成！")
print(f"文件数量：{len(files)}")
print(f"有效路径：{len(result_df)}")
print(
    f"Pathway数量："
    f"{result_df['Pathway_Group'].nunique()}"
)
print(f"输出文件：{output_excel}")

if failed_files:
    print("\n失败文件：")

    for file, error in failed_files:
        print(f"{file}: {error}")

Processing: 100%|██████████| 8/8 [00:00<00:00, 146.76it/s]



处理完成！
文件数量：8
有效路径：543
Pathway数量：19
输出文件：J:\BLA_three_types\csv_PFC_ways_0920\PFC_Pathway_Summary.xlsx
